# Coastal flood step 11: min/max all-sector avoided EAD map comparison

Builds a single comparison figure with **minimum** and **maximum** scenarios shown one above the other using a **fixed shared color range**.

Outputs:
- comparison map PNG (two panels, shared colorbar)
- scenario outlier tables (for auditing highest positive/negative values)


In [ ]:
from pathlib import Path
import numpy
import pandas
import geopandas
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

pandas.set_option('display.max_columns', 200)
pandas.set_option('display.width', 200)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)


In [ ]:
# Core paths
base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
network_csv = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'
shared_intersections_path = base_path / 'dphil_paper_3/results/01_hazard_infrastructure_network_intersections/coastal_flood_network_intersections'
jamaica_boundary_path = base_path / 'dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg'

scenario_results = {
    'minimum': base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario',
    'maximum': base_path / 'dphil_paper_3/results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario',
}

# Cache controls for prebuilt scenario map layers.
# Set to True only when source inputs changed and you need to rebuild.
rebuild_cached_layers = True
cache_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/maps_min_max_comparison/cache'
cache_dir.mkdir(parents=True, exist_ok=True)

for name, p in scenario_results.items():
    if not p.exists():
        raise FileNotFoundError(f'Missing scenario folder: {p}')
if not network_csv.exists():
    raise FileNotFoundError(f'Missing network CSV: {network_csv}')
if not jamaica_boundary_path.exists():
    raise FileNotFoundError(f'Missing Jamaica boundary file: {jamaica_boundary_path}')

print('Scenario folders:')
for name, p in scenario_results.items():
    print('-', name, '->', p)
print('Cache folder:', cache_dir)
print('Rebuild cached layers:', rebuild_cached_layers)


In [ ]:
# Shared network metadata used for joining assets to split geometries
network_details = pandas.read_csv(network_csv)
required_cols = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_cols = [c for c in required_cols if c not in network_details.columns]
if missing_cols:
    raise KeyError(f'Missing required columns in network csv: {missing_cols}')

network_map_details = network_details[required_cols].drop_duplicates().copy()
display(network_map_details.head())
print(f'Network layer mappings: {len(network_map_details):,}')


In [ ]:
def build_scenario_map_layer(results_path: Path, scenario_name: str) -> geopandas.GeoDataFrame:
    asset_file = results_path / 'damage_estimates/coastal_ead_asset_level_usd.csv'
    if not asset_file.exists():
        raise FileNotFoundError(f'Missing asset EAD file for {scenario_name}: {asset_file}')

    asset_ead = pandas.read_csv(asset_file)
    map_layers = []
    missing_split_files = []

    for row in network_map_details.itertuples(index=False):
        split_file = shared_intersections_path / f"{row.asset_gpkg}_splits__coastal_flood_rasters_for_intersections__{row.asset_layer}.geoparquet"
        if not split_file.exists():
            missing_split_files.append(str(split_file))
            continue

        split_geom = geopandas.read_parquet(split_file)
        if split_geom.crs is not None:
            split_geom = split_geom.to_crs('EPSG:3448')
        if row.asset_id_column not in split_geom.columns:
            continue

        split_geom = split_geom[[row.asset_id_column, 'geometry']].copy()
        split_geom = geopandas.GeoDataFrame(split_geom, geometry='geometry', crs=split_geom.crs)

        ead_subset = asset_ead.loc[
            (asset_ead['Asset'] == row.asset_gpkg) & (asset_ead['Layer'] == row.asset_layer),
            ['Asset_ID', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']
        ].copy()
        if ead_subset.empty:
            continue

        split_geom['_join_id'] = split_geom[row.asset_id_column].astype(str)
        ead_subset['_join_id'] = ead_subset['Asset_ID'].astype(str)

        merged = split_geom.merge(
            ead_subset[['_join_id', 'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD']],
            on='_join_id',
            how='left'
        )

        merged['Scenario'] = scenario_name
        merged['Sector'] = row.sector
        merged['Subsector'] = row.asset_description
        merged['Asset'] = row.asset_gpkg
        merged['Layer'] = row.asset_layer
        merged['Asset_ID'] = merged[row.asset_id_column].astype(str)
        merged['Avoided_EAD_USD'] = merged['Avoided_EAD_USD'].fillna(0.0)

        map_layers.append(merged[[
            'Scenario', 'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
            'Avoided_EAD_USD', 'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'geometry'
        ]])

    if not map_layers:
        raise ValueError(f'No map layers built for scenario: {scenario_name}')

    gdf = geopandas.GeoDataFrame(
        pandas.concat(map_layers, ignore_index=True),
        geometry='geometry',
        crs='EPSG:3448'
    )

    print(f"{scenario_name}: loaded {len(gdf):,} mapped features")
    print(f"{scenario_name}: features by sector")
    display(gdf.groupby('Sector', as_index=False).size())

    if missing_split_files:
        print(f"{scenario_name}: missing split files (skipped): {len(set(missing_split_files))}")

    return gdf


def load_or_build_scenario_map_layer(results_path: Path, scenario_name: str) -> geopandas.GeoDataFrame:
    cache_file = cache_dir / f'{scenario_name}_all_sector_map_layer.geoparquet'

    if cache_file.exists() and not rebuild_cached_layers:
        gdf = geopandas.read_parquet(cache_file)
        if gdf.crs is None:
            gdf = gdf.set_crs('EPSG:3448')
        print(f"{scenario_name}: loaded cached map layer -> {cache_file}")
        print(f"{scenario_name}: cached features = {len(gdf):,}")
        return gdf

    gdf = build_scenario_map_layer(results_path, scenario_name)
    gdf.to_parquet(cache_file)
    print(f"{scenario_name}: wrote cache -> {cache_file}")
    return gdf


In [ ]:
scenario_gdfs = {
    name: load_or_build_scenario_map_layer(path, name)
    for name, path in scenario_results.items()
}

jamaica_boundary = geopandas.read_file(jamaica_boundary_path).to_crs('EPSG:3448')

display(scenario_gdfs['minimum'].head(3))


In [ ]:
# Shared scale configuration for side-by-side comparability
# Option A: keep None to use pooled quantile cap across min+max
# Option B: set a fixed cap in USD, e.g., 150.0
manual_shared_cap_usd = None

display_quantile = 0.995
value_col = 'Avoided_EAD_USD'

pooled_abs = pandas.concat([
    scenario_gdfs['minimum'][value_col].fillna(0.0).abs(),
    scenario_gdfs['maximum'][value_col].fillna(0.0).abs(),
], ignore_index=True)

quantile_cap_usd = float(pooled_abs.quantile(display_quantile))
if manual_shared_cap_usd is None:
    shared_cap_usd = quantile_cap_usd
else:
    shared_cap_usd = float(abs(manual_shared_cap_usd))

if shared_cap_usd <= 0:
    shared_cap_usd = float(pooled_abs.max()) if float(pooled_abs.max()) > 0 else 1.0

if shared_cap_usd >= 1e6:
    unit_factor = 1e6
    unit_label = 'USD millions'
elif shared_cap_usd >= 1e3:
    unit_factor = 1e3
    unit_label = 'USD thousands'
else:
    unit_factor = 1.0
    unit_label = 'USD'

shared_cap = shared_cap_usd / unit_factor
norm = TwoSlopeNorm(vmin=-shared_cap, vcenter=0.0, vmax=shared_cap)

print(f'Pooled quantile cap q={display_quantile:.3f}: {quantile_cap_usd:,.2f} USD')
print(f'Shared display cap used: {shared_cap_usd:,.2f} USD ({shared_cap:,.2f} {unit_label})')


In [ ]:
# Styling
cmap = LinearSegmentedColormap.from_list(
    'red_white_green',
    ['#c81e1e', '#ffffff', '#0b8f3f'],
    N=256,
)
pos_outlier_color = '#145a32'  # high avoided
neg_outlier_color = '#8b1a1a'  # high increase
top_labels_per_sign = 6


# Helper to draw one scenario panel
def draw_scenario_panel(ax, gdf, scenario_name):
    panel = gdf.copy()
    vals_usd = panel[value_col].fillna(0.0)

    panel['_plot_val'] = (vals_usd / unit_factor).clip(-shared_cap, shared_cap)
    panel['_is_outlier'] = vals_usd.abs() > shared_cap_usd

    # Base boundary
    jamaica_boundary.boundary.plot(ax=ax, color='#bdbdbd', linewidth=0.45, zorder=1)

    # Plot by geometry type
    geom_type = panel.geometry.geom_type.astype(str)
    polys = panel[geom_type.str.contains('Polygon', na=False)]
    lines = panel[geom_type.str.contains('LineString', na=False)]
    points = panel[geom_type.str.contains('Point', na=False)]

    if not polys.empty:
        polys.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.12, edgecolor='none', alpha=0.9, zorder=2)
    if not lines.empty:
        lines.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, linewidth=0.9, alpha=0.95, zorder=3)
    if not points.empty:
        points.plot(ax=ax, column='_plot_val', cmap=cmap, norm=norm, markersize=16, alpha=0.95, zorder=4)

    # Outlier overlays by sign
    outliers = panel[panel['_is_outlier']].copy()
    pos_outliers = outliers[outliers[value_col] > 0].copy()
    neg_outliers = outliers[outliers[value_col] < 0].copy()

    for subset, color in [(pos_outliers, pos_outlier_color), (neg_outliers, neg_outlier_color)]:
        if subset.empty:
            continue
        subset_geom_type = subset.geometry.geom_type.astype(str)
        subset_polys = subset[subset_geom_type.str.contains('Polygon', na=False)]
        subset_lines = subset[subset_geom_type.str.contains('LineString', na=False)]
        subset_points = subset[subset_geom_type.str.contains('Point', na=False)]

        if not subset_polys.empty:
            subset_polys.boundary.plot(ax=ax, color=color, linewidth=1.15, alpha=0.96, zorder=5)
        if not subset_lines.empty:
            subset_lines.plot(ax=ax, color=color, linewidth=2.0, alpha=0.96, zorder=5)
        if not subset_points.empty:
            subset_points.plot(ax=ax, color=color, markersize=38, alpha=0.96, zorder=5)

    # Top outlier labels (dedup by asset key to reduce clutter)
    key_cols = ['Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID']
    label_source = outliers.assign(_abs_val=outliers[value_col].abs()).sort_values('_abs_val', ascending=False)
    label_source = label_source.drop_duplicates(subset=key_cols, keep='first')

    top_pos = label_source[label_source[value_col] > 0].head(top_labels_per_sign).copy()
    top_neg = label_source[label_source[value_col] < 0].head(top_labels_per_sign).copy()
    label_df = pandas.concat([top_pos, top_neg], ignore_index=True)

    if not label_df.empty:
        label_points = label_df.geometry.representative_point()
        for (_, row), pt in zip(label_df.iterrows(), label_points):
            v_scaled = row[value_col] / unit_factor
            label_color = pos_outlier_color if row[value_col] > 0 else neg_outlier_color
            ax.text(
                pt.x,
                pt.y,
                f"{v_scaled:+,.1f}",
                fontsize=7,
                color=label_color,
                ha='left',
                va='bottom',
                zorder=6,
                bbox={'facecolor': 'white', 'alpha': 0.80, 'edgecolor': label_color, 'pad': 0.35}
            )

    # Tight map extent to Jamaica
    minx, miny, maxx, maxy = jamaica_boundary.total_bounds
    pad_x = (maxx - minx) * 0.005
    pad_y = (maxy - miny) * 0.003
    ax.set_xlim(minx - pad_x, maxx + pad_x)
    ax.set_ylim(miny - pad_y, maxy + pad_y)

    n_out = int(panel['_is_outlier'].sum())
    ax.set_title(f"{scenario_name.capitalize()} scenario | outliers: {n_out:,}", fontsize=12)
    ax.set_axis_off()

    # Scenario outlier table for export
    outlier_table = label_source.copy()
    if not outlier_table.empty:
        rp = outlier_table.geometry.representative_point()
        outlier_table['label_x'] = rp.x
        outlier_table['label_y'] = rp.y
        outlier_table['Outlier_Sign'] = numpy.where(outlier_table[value_col] > 0, 'avoided_positive', 'increase_negative')

    return outlier_table


In [ ]:
# Build final min-vs-max figure (two maps, one fixed shared scale)
fig, axes = plt.subplots(2, 1, figsize=(11, 12.6), sharex=True, sharey=True)

outlier_tables = {}
outlier_tables['minimum'] = draw_scenario_panel(axes[0], scenario_gdfs['minimum'], 'minimum')
outlier_tables['maximum'] = draw_scenario_panel(axes[1], scenario_gdfs['maximum'], 'maximum')

sm = ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# Shared horizontal colorbar for both panels
cax = fig.add_axes([0.16, 0.045, 0.68, 0.018])
cbar = fig.colorbar(sm, cax=cax, orientation='horizontal')
cbar.set_label(
    f'Avoided EAD ({unit_label}), fixed shared range +/-{shared_cap:,.2f} | '
    'Red=increase, White=no change, Green=avoided'
)

fig.suptitle('Total avoided EAD locations across all sectors', fontsize=15, y=0.975)
fig.subplots_adjust(top=0.955, bottom=0.08, hspace=0.01)

out_dir = base_path / 'dphil_paper_3/results_coastal_scenario_comparison/maps_min_max_comparison'
out_dir.mkdir(parents=True, exist_ok=True)

map_png = out_dir / 'total_avoided_ead_locations_all_sectors_min_vs_max.png'
fig.savefig(map_png, dpi=300, bbox_inches='tight')
print('Saved map:', map_png)
plt.show()

# Save outlier tables
for scenario_name, tbl in outlier_tables.items():
    out_csv = out_dir / f'outliers_all_sectors_{scenario_name}_shared_q{int(display_quantile*1000)}.csv'
    if tbl is None or tbl.empty:
        pandas.DataFrame(columns=['Sector','Subsector','Asset','Layer','Asset_ID']).to_csv(out_csv, index=False)
    else:
        keep_cols = [
            'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID', value_col,
            'EAD_With_Mangroves_USD', 'EAD_Without_Mangroves_USD', 'Outlier_Sign', 'label_x', 'label_y'
        ]
        tbl[keep_cols].to_csv(out_csv, index=False)
    print('Saved outliers:', out_csv)


In [ ]:
# Quick summary table for reporting
summary_rows = []
for scenario_name, gdf in scenario_gdfs.items():
    vals = gdf[value_col].fillna(0.0)
    n = len(vals)
    n_out = int((vals.abs() > shared_cap_usd).sum())
    summary_rows.append({
        'Scenario': scenario_name,
        'Features': n,
        'Outliers_Count': n_out,
        'Outliers_Percent': 100.0 * n_out / n if n else numpy.nan,
        'Max_Abs_Avoided_EAD_USD': float(vals.abs().max()) if n else numpy.nan,
    })

summary_df = pandas.DataFrame(summary_rows)
display(summary_df)
print(f'Shared cap (USD): {shared_cap_usd:,.2f}')
print(f'Units shown on colorbar: {unit_label}')
